# 02 - 训练数据准备

完成数据加载、ChatML格式转换、合成数据生成和数据集划分。

## 2.1 加载原始数据

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from utils.data_utils import load_raw_data, create_sample_data
from config import DATA_DIR, DATA_CONFIG

# 加载原始数据
raw_data = load_raw_data(DATA_CONFIG['raw_data_dir'])

print(f"已加载 {len(raw_data)} 个数据子集:")
for name, samples in raw_data.items():
    print(f"  {name}: {len(samples)} 条样本")

## 2.2 查看示例数据

In [ ]:
# 查看第一条样本
first_subset = list(raw_data.keys())[0]
first_sample = raw_data[first_subset][0]

print(f"子集: {first_subset}")
print("\n问题:")
print(first_sample['instruction'])
print("\n回答:")
print(first_sample['output'][:500] + '...')

## 2.3 转换为ChatML格式

In [ ]:
from utils.data_utils import convert_to_chatml
from utils.training_utils import load_model_and_tokenizer
from config import DATA_CONFIG, BASE_MODEL, MODELS_DIR
import os

# 加载tokenizer以使用官方chat template (推荐)
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
_, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,  # 评测/数据处理不需要量化
    device_map='cpu',  # 数据处理用CPU即可
)

# 转换为ChatML格式 (使用tokenizer.apply_chat_template)
dataset = convert_to_chatml(
    data=raw_data,
    tokenizer=tokenizer,  # 传入tokenizer以确保格式一致性
    system_message=DATA_CONFIG['chatml']['system_message']
)

# 清理显存
import torch
torch.cuda.empty_cache()

print('数据集划分:')
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} 条")
    avg_len = sum(len(s['text']) for s in split_data) / len(split_data)
    print(f"    平均长度: {avg_len:.0f} 字符")

## 2.4 查看ChatML格式示例

In [ ]:
# 查看第一条ChatML格式数据
sample = dataset['train'][0]
print("ChatML格式示例:")
print("="*60)
print(sample['text'][:1000])
print("...")
print("="*60)

## 2.5 数据验证

In [ ]:
from utils.data_utils import validate_chatml_format

# 验证数据格式
report = validate_chatml_format(dataset)

print(f"总样本数: {report['total_samples']}")
print(f"有效样本: {report['valid_samples']}")
print(f"无效样本: {report['invalid_samples']}")

if report['stats']:
    print(f"\n统计信息:")
    print(f"  平均长度: {report['stats']['avg_length']:.0f}")
    print(f"  最大长度: {report['stats']['max_length']}")
    print(f"  最小长度: {report['stats']['min_length']}")

if report['issues']:
    print(f"\n发现问题 ({len(report['issues'])} 个):")
    for issue in report['issues'][:5]:
        print(f"  - {issue}")

## 2.6 保存处理后的数据集

In [ ]:
from utils.data_utils import save_dataset

# 保存数据集
processed_dir = DATA_CONFIG['processed_data_dir']
save_dataset(dataset, processed_dir)

print(f"\n数据集已保存到: {processed_dir}")
print("文件列表:")
import os
for f in os.listdir(processed_dir):
    print(f"  {f}")

---

## 下一步

数据准备完成！接下来请打开: **03_model_benchmark.ipynb** 进行模型评测